In [1]:
import numpy as np 
from matio import load_from_mat
from pathlib import Path
import pandas as pd
import warnings

from datetime import timedelta
from glob import glob

In [2]:
def readCSVFile(filepath):
    T = pd.read_csv(filepath)
    
    return T 

In [3]:
def unitConverter(fileUnits):
    if "knots" in fileUnits: 
        convertFactor = 0.51444448824222
    elif "cm/sec" in fileUnits: 
        convertFactor = 0.01 
    else: 
        warnings.warn("File defines water speed neither in knots or cm/s. Conversion factor of 1 is given by default. Please check file's units.") 
        convertFactor = 1
        
    return convertFactor 

In [4]:
def extractData(T):
    dateData = pd.to_datetime(T.iloc[:,0]) 
    numericData = T.iloc[:,1:] 

    return dateData, numericData 

In [5]:
def extractMetaData(T, filename):  
    VarNames = list(T.columns) 
    
    if not (any('Speed' in name for name in VarNames)): 
        raise ValueError("%s does not contain 'Speed' column." %(filename))

    fileUnits = ""
    fileDepth = ""
    
    for name in VarNames:
        if 'Speed' in name: 
            tempName = name[name.index("Speed")+len("Speed "):] 
            
            for i in range(len(tempName)):
                if tempName[i].isalpha():
                    fileUnits += tempName[i]

            if ('cm' in fileUnits) and ('s' in fileUnits): 
                fileUnits = fileUnits[:fileUnits.index('cm') + len('cm')] + '/' + fileUnits[fileUnits.index('s'):] 
            
            fileDepth = filename[filename.index("_")+1:filename.index("-")]
            break 

    return Meta(VarNames, fileUnits, fileDepth)

In [6]:
def valVarNames(current, previous, filePrev, fileCurr): 
    if not current == previous:
        raise TypeError(f"Column mismatch between {filePrev}s and {fileCurr}s.")

In [7]:
def initializeTimeSeries(dateData, numericData):
    dateDiff = []  

    for i in range(len(dateData)-1):
        dateDiff.append(dateData[i+1]-dateData[i]) 

        if not dateDiff[i] == dateDiff[0]:
            warnings.warn("The is uneven date interval within current dataset, and so first interval is chosen by default.")
            break 

    dateInterval = dateDiff[0]

    dateOut = dateData 
    dataOut = numericData
    dateOut_unique = dateData
    uniquedatetime_Length = len(dateData)

    return dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique

In [8]:
def processTimeSeries(prev, dateData, numericData, fileDepth, fileCurrName, filePrevName, dateInterval): 
    if (dateData[0] > prev.dateData.iloc[-1]) and (abs(dateInterval - (dateData[0] - prev.dateData.iloc[-1])) < timedelta(seconds = 1)) and (fileDepth == prev.fileDepth):
        uniquedatetime = np.unique(pd.concat([prev.dateData, dateData], axis = 0))
        uniquedatetime_Length = len(uniquedatetime) 

        if not uniquedatetime_Length == prev.uniquedatetime_Length: 
            timeDiff = abs(uniquedatetime_Length - prev.uniquedatetime_Length) 
            timeDiff_datetime = timeDiff * dateInterval 
            warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}.\nThe previous file ranges from {prev.dateData[0]} to {prev.dateData.iloc[-1]}.\nThe current file ranges from {dateData[0]} to {dateData.iloc[-1]}.\nThis number of entries correspond to a time difference of {timeDiff_datetime}.")
            user_concatenate = input("Do you approve the concatenation of these datasets? Please write '1' to approve or '0' to disapprove: ")

            if user_concatenate == '1': 
                dateOut_unique = uniquedatetime[0:uniquedatetime_Length] 
           
            else:
                dateOut_unique = uniquedatetime[0:prev.uniquedatetime_Length] 

        dateOut = uniquedatetime 
        
        dataOut = pd.concat([prev.numericData, numericData], axis=0, ignore_index=True) 
            
    elif dateData.equals(prev.dateData): 
        dateOut = prev.dateData
        dataOut = numericData

    elif (not dateData.equals(prev.dateData)) and (fileDepth == prev.fileDepth):
        warnings.warn(f"Date mismatch between files {filePrevName} and {fileCurrName}.") 
        dateOut = dateData
        dataOut = numericData 

    else: 
        dateOut = dateData 
        dataOut = numericData 

    try: 
        uniquedatetime_Length
    except NameError: 
        uniquedatetime_Length = prev.uniquedatetime_Length 

    try: 
        dateOut_unique 
    except NameError: 
        dateOut_unique = prev.dateOut_unique 

    return dateOut, dateOut_unique, dataOut, uniquedatetime_Length

In [9]:
def extractSiteID(filenames): 
    siteName = []
    siteNum = [] 
    siteID = ""
    
    temp = ""
    check = True 

    for i in range(len(filenames)):
        for j in range(len(filenames[i])): 
            if filenames[i][j].isalpha(): 
                temp += filenames[i][j]  
            else:
                break 
        siteName.append(temp) 
        temp = "" 
    
    for i in range(len(filenames)): 
        for j in range(len(filenames[i])): 
            if filenames[i][j + len(siteName[i])].isdigit(): 
                temp += filenames[i][j + len(siteName[i])]  
            else:
                break 
        siteNum.append(temp) 
        temp = "" 
    
    for i in range(len(siteName)-1): 
        if not((siteName[i] == siteName[i+1]) and (siteNum[i] == siteNum[i+1])):
            warnings.warn("There is mismatch of site ID within provided data.");
            print("User-defined ID requested for plotting: ")
            siteID = inputID()
            check = False 
            break
    
    if check: 
        siteID = siteName[0] + siteNum[0]
    
    return siteID

In [10]:
def inputID(): 
    user_decision = input('Do you want to continue by defining the site ID? (Y/N): ') 
    
    if user_decision == 'N': 
        print('Exiting from FolderReadCSV. Recommendation to revise data in files.')
        return 
    
    siteID = input("\n Please define the site ID to appear in plots (e.g. (LIS1001)): ") 
    print('Continuing with user') 
    return siteID 

In [11]:
def findDepth(filenames, files_num): 
    fileDepths = [] 
    depthUnits = [] 
    depth_units = "" 
    temp = ""

    for i in range(len(filenames)): 
        extract = filenames[i] 
        fileDepths.append(extract[extract.index("_")+1:extract.index("-")]) 
    
    for i in range(len(fileDepths)): 
        for j in range(len(fileDepths[i])):  
            if fileDepths[i][j].isalpha():
                temp += fileDepths[i][j]  
        depthUnits.append(temp) 
        temp = "" 

    for i in range(len(depthUnits) - 1): 
        if not (depthUnits[i] == depthUnits[i + 1]): 
            raise ValueError("waterDepth:incorrectFormat","Error in file format. \nFile format must list same units after site ID. \nAcceptable formats are the following: \nLIS1001_05m76cm \nLIS1001_18ft09df")
    
    depth_units = depthUnits[0][0] 

    depthDigits = []
    waterDepth = [] 

    for i in range(len(fileDepths)): 
        for j in range(len(fileDepths[i])):  
            if fileDepths[i][j].isdigit(): 
                temp += fileDepths[i][j] 
        depthDigits.append(temp) 
        temp = "" 

    for i in range(len(depthDigits)):
        waterDepth.append(depthDigits[i][:2] + "." + depthDigits[i][2:]) 

    waterDepth = np.unique(waterDepth) 

    return waterDepth, depth_units 

In [12]:
def alignDataLengths(dataCells, targetLength): 
    for i in reversed(range(len(dataCells))): #iterating backwards to avoid del errors 
        #removed isempty() check 

        row, col = np.shape(dataCells[i]) 

        if row < targetLength: 
            del dataCells[i]  
            
        elif row > targetLength: 
            dataCells[i] = dataCells[i].iloc[0:targetLength]

    return dataCells

In [13]:
class Meta: 
    def __init__(self, VarNames, fileUnits, fileDepth): 
        self.VarNames = VarNames
        self.fileUnits = fileUnits
        self.fileDepth = fileDepth

In [14]:
class Prev(): 
    def __init__(self):
        pass 

    def update(self, meta, dateData, numericData, uniquedatetime_Length):
        self.VarNames = meta.VarNames
        self.fileDepth = meta.fileDepth
        self.dateData = dateData
        self.numericData = numericData
        self.uniquedatetime_Length = uniquedatetime_Length 

        return self 

In [15]:
def FolderReadCSV(folderpath): 
    # preprocessing step 1
    files = glob(folderpath)
    files_num = len(files) 

    dataCells = [] 
    dateCells = []
    filenames = []

    prev = Prev() 

    for i in range(files_num):  
        filepath = files[i] 
        filenames.append(files[i][-30:]) #extracts the name of each file from path  

        #read CSV file into a table 
        T = readCSVFile(filepath) 

        #extract metadata 
        meta = extractMetaData(T, filenames[i]) 

        if (i > 0) and (convertFactor in locals()):
            valVarNames(meta.VarNames, prev.VarNames, filenames[i-1], filenames[i]) 
        else:
            convertFactor = unitConverter(meta.fileUnits) 

        dateData, numericData = extractData(T) 

        if (i > 0) and (hasattr(prev, "dateData")): 
            dateOut, dateOut_unique, dataOut, uniquedatetime_Length = processTimeSeries(prev, dateData, numericData, meta.fileDepth, filenames[i], filenames[i-1], dateInterval)
            if not (np.array_equal(dateOut_unique, prev.dateOut_unique)): 
                warnings.warn("There is mismatch between unique dates.") 
                dateOut_uniqueSwitch = input("Write '1' for the new set of dates, or '0' to keep the old set of dates: ") 
                
                if dateOut_uniqueSwitch == '1': 
                    prev.dateOut_unique = dateOut_unique
                elif dateOut_uniqueSwitch == '0': 
                    dateOut_unique = prev.dateOut_unique 
                else: 
                    raise ValueError("Invalid input. Input must be numeric '1' or '0'.") #add this to first instance of input() as well
        else: 
            dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique = initializeTimeSeries(dateData, numericData)
            prev.dateOut_unique = dateOut_unique 
            
        #assignment
        if (i % 2 == 1): 
            dateCells.append(dateOut) 
            dataCells.append(dataOut) 

        #update of prev object 
        prev = prev.update(meta, dateData, numericData, uniquedatetime_Length) 

    #preprocessing step 2
    VarNames = prev.VarNames

    #extract site ID
    siteID = extractSiteID(filenames) 
    
    #gather water depths into list
    seadepths, depth_units = findDepth(filenames, files_num) 

    #daytime manipulation 
    DMY = dateOut_unique 
    dataCells = alignDataLengths(dataCells, len(DMY)) 
    return dataCells, dateCells, files_num, VarNames, DMY, dateInterval, convertFactor, siteID, seadepths, depth_units

In [16]:
dataCells, dateCells, files_num, VarNames, DMY, dateInterval, convertFactor, siteID, seadepths, depth_units = FolderReadCSV('C:\\Users\\ehsia\\Desktop\\SBU\\Tidal Data\\*.csv')

C:\Users\ehsia\AppData\Local\Temp\ipykernel_11812\1217148185.py:9: UserWarning: There is a difference of 4265 date entries between files LIS1016_05m76cm-2010-07-10.csv and LIS1016_05m76cm-2010-06-09.csv.
The previous file ranges from 2010-06-09 20:12:00 to 2010-07-09 23:54:00.
The current file ranges from 2010-07-10 00:00:00 to 2010-07-27 18:24:00.
This number of entries correspond to a time difference of 17 days 18:30:00.
  warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}.\nThe previous file ranges from {prev.dateData[0]} to {prev.dateData.iloc[-1]}.\nThe current file ranges from {dateData[0]} to {dateData.iloc[-1]}.\nThis number of entries correspond to a time difference of {timeDiff_datetime}.")


Do you approve the concatenation of these datasets? Please write '1' to approve or '0' to disapprove:  1


C:\Users\ehsia\AppData\Local\Temp\ipykernel_11812\787101335.py:32: UserWarning: There is mismatch between unique dates.
  warnings.warn("There is mismatch between unique dates.")


Write '1' for the new set of dates, or '0' to keep the old set of dates:  1


C:\Users\ehsia\AppData\Local\Temp\ipykernel_11812\1217148185.py:9: UserWarning: There is a difference of 1 date entries between files LIS1016_25m76cm-2026-07-10.csv and LIS1016_25m76cm-2010-06-09.csv.
The previous file ranges from 2010-06-09 20:12:00 to 2010-07-09 23:54:00.
The current file ranges from 2010-07-10 00:00:00 to 2010-07-27 18:30:00.
This number of entries correspond to a time difference of 0 days 00:06:00.
  warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}.\nThe previous file ranges from {prev.dateData[0]} to {prev.dateData.iloc[-1]}.\nThe current file ranges from {dateData[0]} to {dateData.iloc[-1]}.\nThis number of entries correspond to a time difference of {timeDiff_datetime}.")


Do you approve the concatenation of these datasets? Please write '1' to approve or '0' to disapprove:  9


In [17]:
dataCells

[       Speed (cm/sec)  Dir (true)
 0                43.0         153
 1                33.7         163
 2                31.9         173
 3                32.5         186
 4                27.9         192
 ...               ...         ...
 11498            49.0          37
 11499            58.1          46
 11500            60.5          49
 11501            67.0          47
 11502            23.1         121
 
 [11503 rows x 2 columns],
        Speed (cm/sec)  Dir (true)
 0                47.1         156
 1                35.3         169
 2                34.6         188
 3                32.0         194
 4                31.4         197
 ...               ...         ...
 11498            50.7          51
 11499            55.6          46
 11500            59.4          53
 11501            69.0          51
 11502           131.9         206
 
 [11503 rows x 2 columns],
        Speed (cm/sec)  Dir (true)
 0                43.2         165
 1                31.8         1

In [18]:
DMY

array(['2010-06-09T20:12:00.000000000', '2010-06-09T20:18:00.000000000',
       '2010-06-09T20:24:00.000000000', ...,
       '2010-07-27T18:12:00.000000000', '2010-07-27T18:18:00.000000000',
       '2010-07-27T18:24:00.000000000'],
      shape=(11503,), dtype='datetime64[ns]')

In [19]:
len(DMY)

11503

In [21]:
import math

In [47]:
def extractCosineTideParams(DMY, velSigned): 
    for i in reversed(range(len(DMY))): 
        if not (np.issubdtype(DMY[i], np.datetime64)) and math.isnan(velSigned[i]):
            del DMY[i] 
            del velSigned[i]

    t = (DMY - DMY[0]).astype('timedelta64[s]')

    return t

In [48]:
T = extractCosineTideParams(DMY, DMY)

In [49]:
T

array([      0,     360,     720, ..., 4140000, 4140360, 4140720],
      shape=(11503,), dtype='timedelta64[s]')